# 04 — Fine-tuning MRL y generación de moléculas EGFR-like

Este notebook hace **solo** la parte generativa:

1. Carga el CSV mínimo `model_procesed_dataset.csv` con SMILES activos frente a EGFR.
2. Carga un modelo preentrenado de MRL.
3. Hace fine-tuning supervisado epoch a epoch.
4. Después de cada epoch genera una muestra de control y mide:
   - validez RDKit,
   - unicidad de SMILES raw,
   - unicidad canónica válida,
   - novedad exacta frente al training set.
5. Guarda un checkpoint por epoch con métricas, parámetros e hiperparámetros.
6. Permite elegir la mejor epoch.
7. Carga el checkpoint elegido y genera el conjunto final de moléculas.

**Decisión clave:** este notebook no predice pIC50, no ejecuta ADMET-AI y no hace selección Pareto. Eso va en el notebook siguiente. Aquí se cierra la etapa: *modelo generativo → moléculas generadas válidas y nuevas*.

## 1. Idea técnica y química

El fine-tuning del generador de SMILES no aprende propiedades químicas explícitas como QED, TPSA o pIC50. Aprende una distribución de secuencias SMILES a partir de moléculas activas.

Por eso, el CSV de entrenamiento debe ser mínimo:

```text
smiles
```

Los descriptores químicos y las predicciones de actividad pertenecen a una fase posterior de scoring y filtrado. Meter descriptores como columnas en el fine-tuning del LSTM no le da más información al modelo, porque este modelo no es condicional: solo aprende a generar strings SMILES.

El criterio de parada tampoco debe ser solo el `loss`. En modelos generativos, una loss más baja puede significar más adaptación, pero también más riesgo de pérdida de diversidad o generación de moléculas menos válidas. Por eso se monitorizan métricas generativas tras cada epoch.

In [16]:
# =========================
# Imports generales
# =========================
from pathlib import Path
from datetime import datetime
import json
import random
import sys
import shutil

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [17]:
# =========================
# Rutas del proyecto
# =========================
DATA_DIR = Path("../data").resolve()
OUTPUT_DIR = Path("../outputs/mrl").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_DATA_PATH = DATA_DIR / "model_procesed_dataset.csv"
REFERENCE_DATA_PATH = DATA_DIR / "model_preprocesed_dataset.csv"

RAW_OUTPUT_PATH = OUTPUT_DIR / "generated_molecules_raw.csv"
OUTPUT_PATH = OUTPUT_DIR / "generated_molecules.csv"
GENERATION_METADATA_PATH = OUTPUT_DIR / "generation_metadata.json"
MONITORING_HISTORY_PATH = OUTPUT_DIR / "finetuning_monitoring_history.csv"

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = OUTPUT_DIR / "mrl_lstm_egfr_finetuned_best.pt"

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("TRAINING_DATA_PATH:", TRAINING_DATA_PATH)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

DATA_DIR: /home/kluna/TFM/tfm_project/data
OUTPUT_DIR: /home/kluna/TFM/tfm_project/outputs/mrl
TRAINING_DATA_PATH: /home/kluna/TFM/tfm_project/data/model_procesed_dataset.csv
CHECKPOINT_DIR: /home/kluna/TFM/tfm_project/outputs/mrl/checkpoints


## 2. Parámetros e hiperparámetros

Usamos un modelo preentrenado de MRL y hacemos fine-tuning con learning rate bajo. La idea no es entrenar agresivamente, sino desplazar el modelo hacia el espacio químico de inhibidores EGFR manteniendo validez y diversidad.

- `MAX_FINETUNE_EPOCHS` se interpreta como máximo, no como obligación.
- Se genera una muestra de control tras cada epoch.
- Cada checkpoint queda guardado para poder escoger la mejor epoch sin reentrenar.

In [18]:
# =========================
# Fine-tuning MRL
# =========================
PRETRAINED_MODEL = "LSTM_LM_Small_ZINC_NC"
DROP_SCALE = 0.30
LEARNING_RATE = 5e-5
FINETUNE_BATCH_SIZE = 32
MAX_FINETUNE_EPOCHS = 8

# =========================
# Quality check tras cada epoch
# =========================
N_EVAL_SMILES = 4_000
EVAL_BATCH_SIZE = 512
EVAL_MAX_SMILES_LENGTH = 90

# =========================
# Generación final
# =========================
N_RAW_SMILES_TO_SAMPLE = 10_000
GENERATION_BATCH_SIZE = 512
MAX_SMILES_LENGTH = 120
EXCLUDE_TRAINING_SET_HITS = True

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "pretrained_model": PRETRAINED_MODEL,
    "drop_scale": DROP_SCALE,
    "learning_rate": LEARNING_RATE,
    "finetune_batch_size": FINETUNE_BATCH_SIZE,
    "max_finetune_epochs": MAX_FINETUNE_EPOCHS,
    "n_eval_smiles": N_EVAL_SMILES,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "eval_max_smiles_length": EVAL_MAX_SMILES_LENGTH,
    "n_raw_smiles_to_sample": N_RAW_SMILES_TO_SAMPLE,
    "generation_batch_size": GENERATION_BATCH_SIZE,
    "max_smiles_length": MAX_SMILES_LENGTH,
    "exclude_training_set_hits": EXCLUDE_TRAINING_SET_HITS,
}

CONFIG

{'run_id': '20260505_183148',
 'seed': 42,
 'pretrained_model': 'LSTM_LM_Small_ZINC_NC',
 'drop_scale': 0.3,
 'learning_rate': 5e-05,
 'finetune_batch_size': 32,
 'max_finetune_epochs': 8,
 'n_eval_smiles': 4000,
 'eval_batch_size': 512,
 'eval_max_smiles_length': 90,
 'n_raw_smiles_to_sample': 10000,
 'generation_batch_size': 512,
 'max_smiles_length': 120,
 'exclude_training_set_hits': True}

## 3. Imports químicos y MRL

Se usan dos funciones del proyecto:

- `mol_from_smiles(smiles)`
- `canonicalize_smiles(smiles)`

El entrenamiento asume que `model_procesed_dataset.csv` ya contiene SMILES canónicos y activos. Aun así, en la generación sí canonicalizamos de nuevo, porque los SMILES generados son strings producidos por el modelo y pueden ser inválidos o representaciones alternativas de la misma molécula.

In [19]:
# =========================
# RDKit y funciones propias
# =========================
try:
    from rdkit import Chem
except ImportError as e:
    raise ImportError("RDKit no está disponible. Instálalo antes de continuar.") from e

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

try:
    from funciones import mol_from_smiles, canonicalize_smiles
except ImportError as e:
    raise ImportError(
        "No pude importar mol_from_smiles/canonicalize_smiles desde ../funciones.py. "
        "Revisa que el archivo exista y que el notebook esté en la carpeta notebooks/."
    ) from e

# =========================
# Torch y MRL
# =========================
try:
    import torch
    torch.manual_seed(SEED)

    from mrl.imports import *
    from mrl.core import *
    from mrl.chem import *
    from mrl.torch_imports import *
    from mrl.torch_core import *
    from mrl.dataloaders import *
    from mrl.g_models.all import *
    from mrl.vocab import *
    from mrl.train.all import *
    from mrl.model_zoo import *
except ImportError as e:
    raise ImportError(
        "MRL no está disponible. Recomendado: entorno limpio compatible con MRL."
    ) from e

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("RDKit OK")
print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
print("Device usado:", DEVICE)

RDKit OK
Torch: 2.11.0+cu130
CUDA disponible: False
Device usado: cpu


## 4. Carga del dataset mínimo

El dataset mínimo debe contener únicamente la columna `smiles`. No se recalculan descriptores aquí porque eso pertenece al scoring posterior.

Aun así, se hacen controles mínimos:

- eliminar nulos/vacíos,
- eliminar duplicados exactos,
- crear `training_set` para detectar copias exactas durante la generación.

In [20]:
if not TRAINING_DATA_PATH.exists():
    raise FileNotFoundError(f"No existe el dataset de entrenamiento: {TRAINING_DATA_PATH}")

df_raw = pd.read_csv(TRAINING_DATA_PATH)

print("Columnas del CSV mínimo:", list(df_raw.columns))
print("Filas iniciales:", len(df_raw))


Columnas del CSV mínimo: ['smiles']
Filas iniciales: 8553


In [21]:
df_train = df_raw[["smiles"]].copy()

training_smiles = df_train["smiles"].tolist()
training_set = set(training_smiles)

print("SMILES únicos usados para fine-tuning:", len(training_smiles))

df_train.head()

SMILES únicos usados para fine-tuning: 8553


,smiles
0,Cn1cnc2cc3ncnc(Nc4cccc(Br)c4)c3cc21
1,Cn1cnc2cc3c(Nc4cccc(Br)c4)ncnc3cc21
2,COc1cc2ncnc(Nc3cccc(Cl)c3F)c2cc1N1CC2(CN(C)C2)...
3,CCCN1CC2(C1)CN(c1cc3c(Nc4cccc(Cl)c4F)ncnc3cc1O...
4,C=CC(=O)Nc1ccc2ncnc(Nc3cc(F)c(Cl)c(Cl)c3)c2c1


## 5. Inicialización del agente MRL

Se carga el modelo preentrenado elegido. El fine-tuning se hará con `update_dataset_from_inputs(...)` y `train_supervised(...)`.

La generación de control usa `batch_sample_and_reconstruct(...)`, que es la forma compacta de pedir al agente que muestree SMILES por batches y los reconstruya como strings.

In [22]:
agent = LSTM_LM_Small_ZINC_NC(
    drop_scale=DROP_SCALE,
    opt_kwargs={"lr": LEARNING_RATE},
)

agent.update_dataset_from_inputs(training_smiles)

print("Agente MRL inicializado:", PRETRAINED_MODEL)

Agente MRL inicializado: LSTM_LM_Small_ZINC_NC


## 6. Función de quality check de generación

Esta función no entrena. Solo pregunta: **¿qué está generando el modelo ahora mismo?**

Mide:

- `percent_valid`: proporción de SMILES interpretables por RDKit.
- `percent_unique_raw`: proporción de strings SMILES crudos únicos.
- `percent_unique_valid`: proporción de moléculas válidas únicas tras canonicalización.
- `percent_novel_vs_training`: moléculas válidas únicas que no están exactamente en el training set.

La métrica de novedad exacta no sustituye a Tanimoto similarity. Solo detecta copias exactas. La similitud estructural se evalúa en el siguiente notebook.

In [23]:
RUN_CHECKPOINT_DIR = CHECKPOINT_DIR / f"run_{RUN_ID}"
RUN_OUTPUT_DIR = OUTPUT_DIR / f"run_{RUN_ID}"

RUN_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MONITORING_HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)

def evaluate_agent_generation(
    agent,
    n_samples: int = 4000,
    batch_size: int = 512,
    max_len: int = 90,
    training_set=None,
) -> tuple[pd.DataFrame, dict]:
    """
    Genera SMILES con el agente MRL y calcula métricas básicas de calidad.
    """
    smiles_raw = agent.batch_sample_and_reconstruct(
        n_samples,
        batch_size,
        max_len,
    )

    rows = []

    for smi in smiles_raw:
        smi = str(smi).strip()
        canonical = canonicalize_smiles(smi)

        rows.append({
            "smiles_raw": smi,
            "smiles": canonical,
            "valid_rdkit": canonical is not None,
        })

    df_eval = pd.DataFrame(rows)

    n_total = len(df_eval)
    n_valid = int(df_eval["valid_rdkit"].sum())

    valid_smiles = df_eval.loc[df_eval["valid_rdkit"], "smiles"]
    n_unique_valid = valid_smiles.nunique()

    metrics = {
        "n_total": n_total,
        "n_valid": n_valid,
        "n_unique_valid": n_unique_valid,
        "percent_valid": n_valid / n_total if n_total else 0,
        "percent_unique_raw": len(set(smiles_raw)) / len(smiles_raw) if smiles_raw else 0,
        "percent_unique_valid": n_unique_valid / n_valid if n_valid else 0,
    }

    if training_set is not None and n_unique_valid > 0:
        unique_valid_set = set(valid_smiles.dropna())
        novel_set = unique_valid_set - training_set

        metrics["n_novel_vs_training"] = len(novel_set)
        metrics["percent_novel_vs_training"] = len(novel_set) / len(unique_valid_set)
    else:
        metrics["n_novel_vs_training"] = np.nan
        metrics["percent_novel_vs_training"] = np.nan

    return df_eval, metrics


def append_epoch_history(row: dict, csv_path: Path) -> None:
    """
    Añade una fila al CSV histórico sin borrar lo anterior.
    Si el archivo no existe, lo crea con headers.
    """

    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)

    df_row = pd.DataFrame([row])

    file_exists_and_not_empty = csv_path.exists() and csv_path.stat().st_size > 0

    df_row.to_csv(
        csv_path,
        mode="a",
        header=not file_exists_and_not_empty,
        index=False,
    )



## 7. Fine-tuning monitorizado y checkpoints

Después de cada epoch:

1. se genera una muestra de control,
2. se calculan métricas de calidad,
3. se guarda el CSV de evaluación de esa epoch,
4. se guarda un checkpoint del modelo,
5. se guarda metadata del entrenamiento y de los hiperparámetros.

Esto permite elegir posteriormente la mejor epoch. No se asume que la última sea la mejor.

In [24]:


history = []

print("=" * 80)
print("Iniciando fine-tuning")
print("RUN_ID:", RUN_ID)
print("Checkpoints en:", RUN_CHECKPOINT_DIR)
print("Evaluaciones en:", RUN_OUTPUT_DIR)
print("Histórico global en:", MONITORING_HISTORY_PATH)
print("=" * 80)

for epoch in range(1, MAX_FINETUNE_EPOCHS + 1):
    print("=" * 80)
    print(f"Fine-tuning epoch {epoch}/{MAX_FINETUNE_EPOCHS}")

    agent.train_supervised(
        FINETUNE_BATCH_SIZE,
        1,
        LEARNING_RATE,
    )

    df_eval, metrics = evaluate_agent_generation(
        agent=agent,
        n_samples=N_EVAL_SMILES,
        batch_size=EVAL_BATCH_SIZE,
        max_len=EVAL_MAX_SMILES_LENGTH,
        training_set=training_set,
    )

    checkpoint_path = RUN_CHECKPOINT_DIR / f"mrl_lstm_egfr_finetuned_epoch_{epoch:02d}.pt"
    eval_output_path = RUN_OUTPUT_DIR / f"generated_eval_epoch_{epoch:02d}.csv"

    epoch_metadata = {
        **CONFIG,
        "run_id": RUN_ID,
        "epoch": epoch,
        "checkpoint_path": str(checkpoint_path),
        "eval_output_path": str(eval_output_path),
        "created_at": datetime.now().isoformat(),
        "n_training_smiles": len(training_smiles),
        "device": str(DEVICE),
        "finetune_batch_size": FINETUNE_BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "n_eval_smiles": N_EVAL_SMILES,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "eval_max_smiles_length": EVAL_MAX_SMILES_LENGTH,
    }

    metrics_with_metadata = {
        **metrics,
        **epoch_metadata,
    }

    history.append(metrics_with_metadata)

    df_eval.to_csv(eval_output_path, index=False)

    torch.save(
        {
            "run_id": RUN_ID,
            "epoch": epoch,
            "model_state_dict": agent.model.state_dict(),
            "metrics": metrics,
            "metadata": epoch_metadata,
        },
        checkpoint_path,
    )

    # --------------------------------------------------------
    # Añadir fila al histórico global
    # --------------------------------------------------------

    append_epoch_history(
        metrics_with_metadata,
        MONITORING_HISTORY_PATH,
    )

    print(
        f"Valid: {metrics['percent_valid']:.3f} | "
        f"Unique raw: {metrics['percent_unique_raw']:.3f} | "
        f"Unique valid canonical: {metrics['percent_unique_valid']:.3f} | "
        f"Novel vs training: {metrics['percent_novel_vs_training']:.3f}"
    )

    print("Checkpoint guardado en:", checkpoint_path)
    print("Evaluación guardada en:", eval_output_path)
    print("Histórico global actualizado en:", MONITORING_HISTORY_PATH)

    display(
        df_eval[df_eval["valid_rdkit"]]
        .drop_duplicates(subset="smiles")
        .head(10)
    )

history_df = pd.DataFrame(history)
history_df

Iniciando fine-tuning
RUN_ID: 20260505_183148
Checkpoints en: /home/kluna/TFM/tfm_project/outputs/mrl/checkpoints/run_20260505_183148
Evaluaciones en: /home/kluna/TFM/tfm_project/outputs/mrl/run_20260505_183148
Histórico global en: /home/kluna/TFM/tfm_project/outputs/mrl/finetuning_monitoring_history.csv
Fine-tuning epoch 1/8


Epoch,Train Loss,Valid Loss,Time
0,0.49713,0.47564,08:24


Valid: 0.970 | Unique raw: 1.000 | Unique valid canonical: 1.000 | Novel vs training: 1.000
Checkpoint guardado en: /home/kluna/TFM/tfm_project/outputs/mrl/checkpoints/run_20260505_183148/mrl_lstm_egfr_finetuned_epoch_01.pt
Evaluación guardada en: /home/kluna/TFM/tfm_project/outputs/mrl/run_20260505_183148/generated_eval_epoch_01.csv
Histórico global actualizado en: /home/kluna/TFM/tfm_project/outputs/mrl/finetuning_monitoring_history.csv


,smiles_raw,smiles,valid_rdkit
0,CC(C)(C)c1ccc(CCNC(=O)CN2CCC3(CCOC3=O)C2)cc1,CC(C)(C)c1ccc(CCNC(=O)CN2CCC3(CCOC3=O)C2)cc1,True
1,CCC(C)CC(C)NC(=O)N1CCn2c(nnc2C(C)(C)C)C1,CCC(C)CC(C)NC(=O)N1CCn2c(nnc2C(C)(C)C)C1,True
2,C=CCC1(NC(=O)Nc2ccc3c(c2)nc(C)n3C(C)CO)CCCC1,C=CCC1(NC(=O)Nc2ccc3c(c2)nc(C)n3C(C)CO)CCCC1,True
3,CCC=C(C)C(=O)NC12CCCC1CN(C(=O)c1cc(C)nn1CC)C2,CCC=C(C)C(=O)NC12CCCC1CN(C(=O)c1cc(C)nn1CC)C2,True
5,CCC(C)(C)C(=O)N(C)CCOCCN(C)C(=O)C1CC1C(=O)OC,CCC(C)(C)C(=O)N(C)CCOCCN(C)C(=O)C1CC1C(=O)OC,True
6,CC1C(=O)NCCN1C(=O)c1csc(-c2cnn(C)c2)n1,CC1C(=O)NCCN1C(=O)c1csc(-c2cnn(C)c2)n1,True
7,CC1(CCC(=O)NCC2CN(Cc3ccccc3)CCCO2)CC1,CC1(CCC(=O)NCC2CN(Cc3ccccc3)CCCO2)CC1,True
8,C#Cc1ccc(NC(=O)CC(C)(C)c2ccccc2C)c(Cl)c1,C#Cc1ccc(NC(=O)CC(C)(C)c2ccccc2C)c(Cl)c1,True
9,N#CCOc1ccccc1C=NN1C(=O)c2ccc(Cl)c(Cl)c2NC1=O,N#CCOc1ccccc1C=Nn1c(=O)[nH]c2c(Cl)c(Cl)ccc2c1=O,True
10,C=CCCCNCC(C1CC1)N(C)C(=O)c1ncn(C(C)(C)C)n1,C=CCCCNCC(C1CC1)N(C)C(=O)c1ncn(C(C)(C)C)n1,True


Fine-tuning epoch 2/8


...

Fine-tuning epoch 8/8


Epoch,Train Loss,Valid Loss,Time
0,0.25835,0.26988,06:31


Valid: 0.888 | Unique raw: 0.995 | Unique valid canonical: 0.994 | Novel vs training: 0.984
Checkpoint guardado en: /home/kluna/TFM/tfm_project/outputs/mrl/checkpoints/run_20260505_183148/mrl_lstm_egfr_finetuned_epoch_08.pt
Evaluación guardada en: /home/kluna/TFM/tfm_project/outputs/mrl/run_20260505_183148/generated_eval_epoch_08.csv
Histórico global actualizado en: /home/kluna/TFM/tfm_project/outputs/mrl/finetuning_monitoring_history.csv


,smiles_raw,smiles,valid_rdkit
0,CC1C(=O)N(C)C(=O)Nc2ncnc(NCc3ccc4c(c3)OCO4)c21,CC1C(=O)N(C)C(=O)Nc2ncnc(NCc3ccc4c(c3)OCO4)c21,True
1,C=CC(=O)Nc1cc(Nc2nc3ccccc3c3nc(SCC(=O)Nc4ccc(C...,C=CC(=O)Nc1cc(Nc2nc3ccccc3c3nc(SCC(=O)Nc4ccc(C...,True
2,C=CC(=O)Nc1cc2c(Nc3ccc(C(=O)NCCOC)cc3)ncnc2cc1OC,C=CC(=O)Nc1cc2c(Nc3ccc(C(=O)NCCOC)cc3)ncnc2cc1OC,True
3,Cc1nc(CN(C)Cc2nc(O)c3ccc(C(F)(F)F)cc3n2)cs1,Cc1nc(CN(C)Cc2nc(O)c3ccc(C(F)(F)F)cc3n2)cs1,True
4,C=CC(=O)Nc1cc(NC(=O)C2CC2c2ccc(F)cc2F)c(Br)cc1Cl,C=CC(=O)Nc1cc(NC(=O)C2CC2c2ccc(F)cc2F)c(Br)cc1Cl,True
5,Cc1ncn(C)c1C(=O)Nc1cc(I)cc([N+](=O)[O-])c1,Cc1ncn(C)c1C(=O)Nc1cc(I)cc([N+](=O)[O-])c1,True
6,Cc1cc(N2CCC(NC(=O)c3ccc(C)c(C(F)(F)F)c3)C2=O)c...,Cc1cc(N2CCC(NC(=O)c3ccc(C)c(C(F)(F)F)c3)C2=O)c...,True
7,COC1=SN(C2CN(c3ncccn3)C2)S/C1=C/C(=O)c1ccc(OC)...,COC1=[SH]N(C2CN(c3ncccn3)C2)S/C1=C/C(=O)c1ccc(...,True
8,COc1cc(Nc2cc3ncnc(Nc4ccc(F)c(Cl)c4)c3n2CC(OC)O...,COc1cc(Nc2cc3ncnc(Nc4ccc(F)c(Cl)c4)c3n2CC(OC)O...,True
9,C=CC(=O)Nc1cccc(-n2c(=O)[nH]c(=O)c(-c3cccc(-c4...,C=CC(=O)Nc1cccc(-n2nc(-c3cccc(-c4cccnc4)c3)c(=...,True


,n_total,n_valid,n_unique_valid,percent_valid,percent_unique_raw,percent_unique_valid,n_novel_vs_training,percent_novel_vs_training,run_id,seed,...,n_raw_smiles_to_sample,generation_batch_size,max_smiles_length,exclude_training_set_hits,epoch,checkpoint_path,eval_output_path,created_at,n_training_smiles,device
0,4000,3880,3880,0.97000,1.00000,1.000000,3880,1.000000,20260505_183148,42,...,10000,512,120,True,1,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T18:41:24.557453,8553,cpu
1,4000,3728,3728,0.93200,1.00000,1.000000,3728,1.000000,20260505_183148,42,...,10000,512,120,True,2,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T18:50:08.604460,8553,cpu
2,4000,3619,3619,0.90475,1.00000,1.000000,3617,0.999447,20260505_183148,42,...,10000,512,120,True,3,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:01:24.422648,8553,cpu
3,4000,3558,3558,0.88950,1.00000,1.000000,3554,0.998876,20260505_183148,42,...,10000,512,120,True,4,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:10:16.632592,8553,cpu
4,4000,3547,3544,0.88675,0.99925,0.999154,3526,0.994921,20260505_183148,42,...,10000,512,120,True,5,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:18:50.210289,8553,cpu
5,4000,3529,3511,0.88225,0.99550,0.994899,3485,0.992595,20260505_183148,42,...,10000,512,120,True,6,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:29:52.576233,8553,cpu
6,4000,3517,3501,0.87925,0.99600,0.995451,3476,0.992859,20260505_183148,42,...,10000,512,120,True,7,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:40:55.228484,8553,cpu
7,4000,3551,3530,0.88775,0.99475,0.994086,3473,0.983853,20260505_183148,42,...,10000,512,120,True,8,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:48:30.349920,8553,cpu


## 8. Elección de la mejor epoch

La mejor epoch no se elige solo por loss. Se elige por equilibrio entre:

- alta validez RDKit,
- alta unicidad canónica,
- alta novedad frente al training set,
- inspección química visual razonable.

La celda siguiente propone una epoch con una regla simple, pero la decisión final puede fijarse manualmente en `BEST_EPOCH`.

In [36]:
df_history = pd.read_csv(MONITORING_HISTORY_PATH)
df_history

,n_total,n_valid,n_unique_valid,percent_valid,percent_unique_raw,percent_unique_valid,n_novel_vs_training,percent_novel_vs_training,run_id,seed,...,n_raw_smiles_to_sample,generation_batch_size,max_smiles_length,exclude_training_set_hits,epoch,checkpoint_path,eval_output_path,created_at,n_training_smiles,device
0,4000,3880,3880,0.97000,1.00000,1.000000,3880,1.000000,20260505_183148,42,...,10000,512,120,True,1,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T18:41:24.557453,8553,cpu
1,4000,3728,3728,0.93200,1.00000,1.000000,3728,1.000000,20260505_183148,42,...,10000,512,120,True,2,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T18:50:08.604460,8553,cpu
2,4000,3619,3619,0.90475,1.00000,1.000000,3617,0.999447,20260505_183148,42,...,10000,512,120,True,3,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:01:24.422648,8553,cpu
3,4000,3558,3558,0.88950,1.00000,1.000000,3554,0.998876,20260505_183148,42,...,10000,512,120,True,4,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:10:16.632592,8553,cpu
4,4000,3547,3544,0.88675,0.99925,0.999154,3526,0.994921,20260505_183148,42,...,10000,512,120,True,5,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:18:50.210289,8553,cpu
5,4000,3529,3511,0.88225,0.99550,0.994899,3485,0.992595,20260505_183148,42,...,10000,512,120,True,6,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:29:52.576233,8553,cpu
6,4000,3517,3501,0.87925,0.99600,0.995451,3476,0.992859,20260505_183148,42,...,10000,512,120,True,7,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:40:55.228484,8553,cpu
7,4000,3551,3530,0.88775,0.99475,0.994086,3473,0.983853,20260505_183148,42,...,10000,512,120,True,8,/home/kluna/TFM/tfm_project/outputs/mrl/checkp...,/home/kluna/TFM/tfm_project/outputs/mrl/run_20...,2026-05-05T19:48:30.349920,8553,cpu


hemos escogido la 3, porque aunque la 1 es la mejor creo que aun no habrá aprendido bien del dataset con solo 1 epoch. En la tercera epoch aun se conserva mas de 90% de validez.

In [ ]:
# Ajusta manualmente si la inspección química justifica otra epoch.
#BEST_EPOCH = suggested_best_epoch
BEST_EPOCH = 3
BEST_CHECKPOINT_PATH = RUN_CHECKPOINT_DIR / f"mrl_lstm_egfr_finetuned_epoch_{BEST_EPOCH:02d}.pt"

if not BEST_CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"No existe el checkpoint seleccionado: {BEST_CHECKPOINT_PATH}")

shutil.copyfile(BEST_CHECKPOINT_PATH, 
                )

print("Checkpoint seleccionado:", BEST_CHECKPOINT_PATH)
print("Modelo best copiado en:", BEST_MODEL_PATH)

Checkpoint seleccionado: /home/kluna/TFM/tfm_project/outputs/mrl/checkpoints/run_20260505_183148/mrl_lstm_egfr_finetuned_epoch_03.pt
Modelo best copiado en: /home/kluna/TFM/tfm_project/outputs/mrl/mrl_lstm_egfr_finetuned_best.pt


## 9. Carga del checkpoint seleccionado

Se reconstruye el modelo preentrenado y se cargan los pesos de la mejor epoch. Esto garantiza que la generación final usa exactamente el checkpoint seleccionado.

In [28]:
agent_best = LSTM_LM_Small_ZINC_NC(
    drop_scale=DROP_SCALE,
    opt_kwargs={"lr": LEARNING_RATE},
)

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    agent_best.model.load_state_dict(checkpoint["model_state_dict"])
    selected_checkpoint_metadata = checkpoint.get("metadata", {})
else:
    agent_best.model.load_state_dict(checkpoint)
    selected_checkpoint_metadata = {}

agent_best.model.eval()

print("Modelo seleccionado cargado desde:", BEST_MODEL_PATH)
print("Metadata checkpoint:", selected_checkpoint_metadata)

Modelo seleccionado cargado desde: /home/kluna/TFM/tfm_project/outputs/mrl/mrl_lstm_egfr_finetuned_best.pt
Metadata checkpoint: {'run_id': '20260505_183148', 'seed': 42, 'pretrained_model': 'LSTM_LM_Small_ZINC_NC', 'drop_scale': 0.3, 'learning_rate': 5e-05, 'finetune_batch_size': 32, 'max_finetune_epochs': 8, 'n_eval_smiles': 4000, 'eval_batch_size': 512, 'eval_max_smiles_length': 90, 'n_raw_smiles_to_sample': 10000, 'generation_batch_size': 512, 'max_smiles_length': 120, 'exclude_training_set_hits': True, 'epoch': 3, 'checkpoint_path': '/home/kluna/TFM/tfm_project/outputs/mrl/checkpoints/run_20260505_183148/mrl_lstm_egfr_finetuned_epoch_03.pt', 'eval_output_path': '/home/kluna/TFM/tfm_project/outputs/mrl/run_20260505_183148/generated_eval_epoch_03.csv', 'created_at': '2026-05-05T19:01:24.422648', 'n_training_smiles': 8553, 'device': 'cpu'}


## 10. Generación final

A partir del checkpoint elegido se genera el conjunto final. Aquí se guardan dos archivos:

- `generated_molecules_raw.csv`: todos los SMILES generados, incluidos inválidos.
- `generated_molecules.csv`: SMILES válidos, canónicos, únicos y opcionalmente sin copias exactas del training set.

Todavía no se calculan descriptores químicos aquí. Eso se hace en el siguiente notebook para mantener clara la frontera entre generación y scoring.

In [30]:
df_generated_raw, generation_metrics = evaluate_agent_generation(
    agent=agent_best,
    n_samples=N_RAW_SMILES_TO_SAMPLE,
    batch_size=GENERATION_BATCH_SIZE,
    max_len=MAX_SMILES_LENGTH,
    training_set=training_set,
)

print("Métricas generación final:")
for key, value in generation_metrics.items():
    print(f"{key}: {value}")

df_generated_raw.to_csv(RAW_OUTPUT_PATH, index=False)
print("Raw output guardado en:", RAW_OUTPUT_PATH)

# Limpieza para dataset generado final

df_generated = (
    df_generated_raw[df_generated_raw["valid_rdkit"]]
    .dropna(subset=["smiles"])
    .drop_duplicates(subset="smiles")
    .reset_index(drop=True)
    .copy()
)

df_generated["in_training_set"] = df_generated["smiles"].isin(training_set)

if EXCLUDE_TRAINING_SET_HITS:
    df_generated = (
        df_generated[~df_generated["in_training_set"]]
        .reset_index(drop=True)
        .copy()
    )

# Metadata mínima como columnas útiles

df_generated.to_csv(OUTPUT_PATH, index=False)

print("Moléculas generadas limpias:", len(df_generated))
print("Output final guardado en:", OUTPUT_PATH)

df_generated.head()

Métricas generación final:
n_total: 10000
n_valid: 9082
n_unique_valid: 9079
percent_valid: 0.9082
percent_unique_raw: 0.9997
percent_unique_valid: 0.999669676282757
n_novel_vs_training: 9068
percent_novel_vs_training: 0.9987884128207952
Raw output guardado en: /home/kluna/TFM/tfm_project/outputs/mrl/generated_molecules_raw.csv
Moléculas generadas limpias: 9068
Output final guardado en: /home/kluna/TFM/tfm_project/outputs/mrl/generated_molecules.csv


,smiles_raw,smiles,valid_rdkit,in_training_set
0,Cc1ccc(C(=O)N2CC3CC2C3NC(=O)Cc2ccoc2)s1,Cc1ccc(C(=O)N2CC3CC2C3NC(=O)Cc2ccoc2)s1,True,False
1,Cc1cc(Br)c2c(c1)CN(C(=O)C1CCc3nc[nH]c3C1)CC2,Cc1cc(Br)c2c(c1)CN(C(=O)C1CCc3nc[nH]c3C1)CC2,True,False
2,Cc1ccc(-c2nc(CC(=O)N3CCN(C(=O)COc4ccc(Cl)cc4)C...,Cc1ccc(-c2nc(CC(=O)N3CCN(C(=O)COc4ccc(Cl)cc4)C...,True,False
3,CC(=O)Nc1ccc2ncnc(Nc3cnn(CCN4CCOCC4)c3)c2c1,CC(=O)Nc1ccc2ncnc(Nc3cnn(CCN4CCOCC4)c3)c2c1,True,False
4,C=CCCC(C)N1CCC2(CCN(C3COC3)C2)CC1,C=CCCC(C)N1CCC2(CC1)CCN(C1COC1)C2,True,False


## 12. Cierre del notebook

Este notebook termina cuando existen estos artefactos:

```text
../outputs/mrl/generated_molecules_raw.csv
../outputs/mrl/generated_molecules.csv
../outputs/mrl/generation_metadata.json
../outputs/mrl/finetuning_monitoring_history.csv
../outputs/mrl/checkpoints/*.pt
../outputs/mrl/mrl_lstm_egfr_finetuned_best.pt
```

El siguiente notebook empieza importando `generated_molecules.csv` y se encarga de descriptores, pIC50, ADMET-AI, similitud, FDA y Pareto.